[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Compression and Archives &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


In [1]:

import zipfile
import gzip
import io
import csv
from pathlib import Path
import shutil

scratch = Path("scratch")
mine = scratch / "mine"
mine.mkdir(parents=True, exist_ok=True)


**1.** Create three text files in `scratch/mine`, then zip them with compression. Print the
archive size against the total of the originals.


In [2]:

(mine / "short.txt").write_text("hello\n", encoding="utf-8")
(mine / "medium.txt").write_text("line\n" * 50, encoding="utf-8")
(mine / "long.txt").write_text("the same words over and over " * 500, encoding="utf-8")

archive = scratch / "mine.zip"

with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(mine.iterdir()):
        z.write(path, arcname=path.name)

originals = sum(p.stat().st_size for p in mine.iterdir())

print("originals:", originals, "bytes")
print("archive:  ", archive.stat().st_size, "bytes")


originals: 14756 bytes
archive:   401 bytes


`arcname=path.name` stores each file under its own name. Without it the archive would contain
`scratch/mine/short.txt`, and unpacking it anywhere would recreate that whole path.


**2.** List the names inside the archive without extracting it.


In [3]:

with zipfile.ZipFile(archive) as z:
    print(z.namelist())


['long.txt', 'medium.txt', 'short.txt']


**3.** Print the original and compressed size of each member, and the ratio.


In [4]:

with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        ratio = info.compress_size / info.file_size if info.file_size else 0
        print(f"{info.filename:<14} {info.file_size:>6} -> {info.compress_size:>6}  {ratio:>5.0%}")


long.txt        14500 ->     79     1%
medium.txt        250 ->     10     4%
short.txt           6 ->      8   133%


`short.txt` is six bytes and comes out larger. The per-member overhead costs more than the
content saves, which is why compressing many tiny files is often pointless.


**4.** Read one member as text, without extracting.


In [5]:

with zipfile.ZipFile(archive) as z:
    print(z.read("short.txt").decode("utf-8"))


hello



`read` gives bytes, so `.decode("utf-8")` is needed. For a large member, `z.open` wrapped in
`io.TextIOWrapper` avoids holding the whole thing in memory.


**5.** Write a CSV, gzip it, and read it back with `csv.DictReader` straight from the `.gz`.


In [6]:

gz = scratch / "rows.csv.gz"

with gzip.open(gz, "wt", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["city", "population"])
    writer.writerow(["Oslo", 709037])
    writer.writerow(["Lima", 9752000])

with gzip.open(gz, "rt", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        print(dict(row))


{'city': 'Oslo', 'population': '709037'}
{'city': 'Lima', 'population': '9752000'}


Both the writing and the reading use `"t"` in the mode. Without it, `csv` receives bytes and
raises, because it works on text.


**6.** For every member, print whether extracting it into `scratch/target` would stay inside
that folder.


In [7]:

destination = (scratch / "target").resolve()

with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        target = (destination / name).resolve()
        print(f"{name:<14} safe: {target.is_relative_to(destination)}")

# and the shape of a name that would fail
print("../escape.txt  safe:",
      (destination / "../escape.txt").resolve().is_relative_to(destination))


long.txt       safe: True
medium.txt     safe: True
short.txt      safe: True
../escape.txt  safe: False


`resolve()` is what makes the check work. It collapses the `..` before the comparison, so the
escaping name is measured by where it actually points rather than by how it is written.

Run this before `extractall` on anything you did not build yourself.


In [8]:

shutil.rmtree(scratch)

print("cleaned up:", not scratch.exists())


cleaned up: True


---

&#8592; **Back to:** [Compression and Archives](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/08-compression-and-archives.ipynb)  &nbsp;&middot;&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
